# Logistics Chatbot


In [2]:
!pip install sentence-transformers transformers gradio torch --quiet

In [6]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import gradio as gr

print("Libraries loaded.")

Libraries loaded.


In [4]:
# Knowledge base — add or edit sentences here to customize the chatbot
knowledge_base = [
    "Lead time is the total time between placing an order and receiving it.",
    "Inventory management ensures products are available while minimizing storage costs.",
    "A warehouse stores goods before distribution to customers.",
    "Transportation management focuses on planning and executing the movement of goods.",
    "Safety stock is extra inventory kept to prevent stockouts.",
    "Cross-docking reduces storage time by transferring goods directly from inbound to outbound vehicles.",
    "The supply chain includes suppliers, manufacturers, warehouses, transporters, and customers.",
    "Last-mile delivery refers to the final step of delivering goods to the customer.",
    "A freight forwarder arranges shipments on behalf of the shipper but does not own the transport vehicles.",
    "A carrier is the company that physically transports the goods using its own vehicles.",
    "Dynamic pricing adjusts freight rates in real time based on demand and available capacity.",
    "Customs clearance is the process of passing goods through customs so they can enter or leave a country.",
    "A bill of lading is a legal document between a shipper and carrier describing the shipment.",
    "Route optimization uses algorithms to find the most efficient delivery routes.",
    "Demand forecasting predicts future customer demand to help plan inventory and capacity.",
    "Predictive maintenance uses sensor data to detect equipment problems before a breakdown occurs.",
    "Cold chain logistics manages temperature-sensitive products throughout the supply chain.",
    "A stockout occurs when demand exceeds available inventory and the product is unavailable.",
    "Vehicle utilization measures how efficiently a truck or fleet is being used relative to its capacity.",
    "An ETA is the estimated time of arrival for a shipment at its destination."
]

print(f"Knowledge base loaded: {len(knowledge_base)} entries.")

Knowledge base loaded: 20 entries.


In [5]:
# Load the embedding model for retrieval
print("Loading embedding model...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
kb_embeddings = embedder.encode(knowledge_base, convert_to_tensor=True)

# Load the language model for generation
print("Loading language model (this takes 1-2 minutes)...")
generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float32
)
print("Models ready.")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading language model (this takes 1-2 minutes)...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Models ready.


In [7]:
def retrieve_context(question):
    q_embedding = embedder.encode(question, convert_to_tensor=True)
    scores = torch.nn.functional.cosine_similarity(
        q_embedding.unsqueeze(0), kb_embeddings
    )
    best_idx = scores.argmax().item()
    return knowledge_base[best_idx]


def logistics_chatbot(question, history):
    context = retrieve_context(question)

    prompt = (
        "<|system|>You are a helpful logistics assistant. "
        "Use the context provided to answer clearly and briefly.</s>"
        f"<|user|>Context: {context}\n\nQuestion: {question}</s>"
        "<|assistant|>"
    )

    result = generator(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    full_text = result[0]["generated_text"]
    answer = full_text.split("<|assistant|>")[-1].strip()

    history.append((question, answer))
    return "", history


print("Functions ready.")

Functions ready.


In [8]:
EXAMPLES = [
    "What is lead time in logistics?",
    "What is the difference between a freight forwarder and a carrier?",
    "What is cross-docking?",
    "How does dynamic pricing work in freight?",
    "What happens when there is a stockout?"
]

with gr.Blocks(theme=gr.themes.Default(font=gr.themes.GoogleFont("Inter"))) as demo:

    gr.Markdown("## Logistics Assistant")
    gr.Markdown("Ask any question about shipments, routes, warehousing, or supply chain decisions.")

    chatbot_ui = gr.Chatbot(height=420, show_label=False)

    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Type your question here...",
            show_label=False,
            scale=8
        )
        send_btn = gr.Button("Send", scale=1, variant="primary")

    gr.Markdown("**Example questions — click to try:**")
    gr.Examples(examples=EXAMPLES, inputs=user_input)

    clear_btn = gr.Button("Clear conversation", variant="secondary")

    send_btn.click(logistics_chatbot, [user_input, chatbot_ui], [user_input, chatbot_ui])
    user_input.submit(logistics_chatbot, [user_input, chatbot_ui], [user_input, chatbot_ui])
    clear_btn.click(lambda: ([], ""), outputs=[chatbot_ui, user_input])

demo.launch(share=True)

/tmp/ipykernel_1279/1901101402.py:9: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(font=gr.themes.GoogleFont("Inter"))) as demo:
/tmp/ipykernel_1279/1901101402.py:14: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(height=420, show_label=False)
/tmp/ipykernel_1279/1901101402.py:14: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(height=420, show_label=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e760a7fa705c7de79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
